# Task 2: Mitsubishi Product Search
## Hybrid Search


Instalasi Library

In [1]:
!pip install PyPDF2 rank_bm25 sentence-transformers pinecone-client pandas numpy python-dotenv langchain-text-splitters

Import Library & Setup Environment

In [2]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)
print("Library dan Environment berhasil disiapkan.")

Library dan Environment berhasil disiapkan.


Ekstraksi Teks dari Folder Dataset

In [ ]:
def extract_text_from_folder(folder_path):
    all_pages_data = []
    pdf_files = list(Path(folder_path).glob("*.pdf"))
    
    if not pdf_files:
        print(f"Peringatan: Tidak ada file PDF ditemukan di folder '{folder_path}'")
        return []

    for pdf_path in pdf_files:
        try:
            reader = PyPDF2.PdfReader(str(pdf_path))
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    
                    clean_text = " ".join(text.split())
                    all_pages_data.append({
                        "text": clean_text,
                        "metadata": {
                            "page": i + 1, 
                            "source": pdf_path.name
                        }
                    })
            print(f"Selesai membaca: {pdf_path.name}")
        except Exception as e:
            print(f"Error membaca {pdf_path.name}: {e}")
            
    return all_pages_data

dataset_folder = "dataset"
raw_docs = extract_text_from_folder(dataset_folder)
print(f"\nTotal halaman yang berhasil diekstrak: {len(raw_docs)}")

Selesai membaca: Mitsubishi-Delica-2015-ID.pdf
Selesai membaca: Mitsubishi-Destinator-2025-ID.pdf
Selesai membaca: Mitsubishi-Eclipse-Cross-2019-ID.pdf
Selesai membaca: Mitsubishi-L100-EV-2024-ID.pdf
Selesai membaca: Mitsubishi-L300-2015-ID.pdf
Selesai membaca: Mitsubishi-L300-2019-ID.pdf
Selesai membaca: Mitsubishi-L300-2021-ID.pdf
Selesai membaca: Mitsubishi-Lancer-2002-ID.pdf
Selesai membaca: Mitsubishi-Outlander-PHEV-2019-ID.pdf
Selesai membaca: Mitsubishi-Outlander-Sport-2018-ID.pdf
Selesai membaca: Mitsubishi-Pajero-Sport-2019-ID.pdf
Selesai membaca: Mitsubishi-Pajero-Sport-2024-ID.pdf
Selesai membaca: Mitsubishi-Pajero-Sport-Elite-Edition-2024-ID.pdf
Selesai membaca: Mitsubishi-Triton-2019-ID.pdf
Selesai membaca: Mitsubishi-Triton-2022-ID.pdf
Selesai membaca: Mitsubishi-Triton-2024-ID.pdf
Selesai membaca: Mitsubishi-Triton-HDX-2025-ID.pdf
Selesai membaca: Mitsubishi-XFC-Concept-2023-ID-.pdf
Selesai membaca: Mitsubishi-XForce-2023-ID.pdf
Selesai membaca: Mitsubishi-Xforce-2025-ID

Chunking Teks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", ".", " "]
)

chunked_docs = []
for doc in raw_docs:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "id": f"{doc['metadata']['source']}_p{doc['metadata']['page']}_c{i}",
            "text": chunk,
            "metadata": doc["metadata"]
        })

print(f"Total chunks yang dihasilkan: {len(chunked_docs)}")

Total chunks yang dihasilkan: 146


Keyword Search (BM25)

In [7]:
corpus_texts = [doc["text"] for doc in chunked_docs]
tokenized_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, top_k=10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({
                "id": chunked_docs[idx]["id"],
                "text": chunked_docs[idx]["text"],
                "metadata": chunked_docs[idx]["metadata"],
                "score": scores[idx]
            })
    return results

print("BM25 siap digunakan.")

BM25 siap digunakan.


Setup Pinecone & Embedding Model

In [ ]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(model_name)

print("Menghasilkan embeddings (mohon tunggu)...")
embeddings = embedding_model.encode(corpus_texts, show_progress_bar=True)

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "mitsubishi-product-search"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=embeddings.shape[1],
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

pinecone_index = pc.Index(index_name)


vectors = []
for i, doc in enumerate(chunked_docs):
    vectors.append({
        "id": doc["id"],
        "values": embeddings[i].tolist(),
        "metadata": {
            "text": doc["text"],
            "page": str(doc["metadata"]["page"]),
            "source": doc["metadata"]["source"]
        }
    })


pinecone_index.upsert(vectors=vectors)
print("Data berhasil di-indeks ke Pinecone Cloud.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7958.15it/s]


Menghasilkan embeddings (mohon tunggu)...


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.34it/s]


Data berhasil di-indeks ke Pinecone Cloud.


Implementasi Hybrid Search (RRF

In [ ]:
def reciprocal_rank_fusion(bm25_res, vector_res, k=60):
    rrf_scores = {}
    
    # Skor dari BM25
    for rank, item in enumerate(bm25_res):
        doc_id = item['id']
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank + 1)
        
    # Skor dari Vector (Pinecone)
    for rank, item in enumerate(vector_res):
        doc_id = item['id']
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank + 1)
        
    # Urutan berdasarkan skor tertinggi
    sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Ambil data asli berdasarkan ID
    lookup = {doc['id']: doc for doc in chunked_docs}
    return [lookup[doc_id] for doc_id, score in sorted_ids]

def hybrid_search_mitsubishi(query, top_k=3):
    # 1. top 10 dari BM25
    bm25_results = search_bm25(query, top_k=10)
    
    # 2. top 10 dari Vector Search
    query_embed = embedding_model.encode(query).tolist()
    vector_res_raw = pinecone_index.query(vector=query_embed, top_k=10, include_metadata=True)
    vector_results = [{"id": match['id']} for match in vector_res_raw['matches']]
    
    # 3. Gabungkan
    final_results = reciprocal_rank_fusion(bm25_results, vector_results)
    return final_results[:top_k]

print("Fungsi Hybrid Search siap digunakan.")

Fungsi Hybrid Search siap digunakan.


Test

In [10]:
test_queries = [
    "Detail spesifikasi Mitsubishi Destinator",
    "Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity yang besar",
    "Mobil untuk perjalanan jauh yang nyaman",
    "Mobil Mitsubishi yang irit bahan bakar",
    "Mobil Mitsubishi hybrid atau electric"
]

print("="*80)
print("HASIL PENCARIAN PRODUK MITSUBISHI (HYBRID SEARCH)")
print("="*80)

for idx, query in enumerate(test_queries, 1):
    print(f"\n[QUERY {idx}]: {query}")
    results = hybrid_search_mitsubishi(query, top_k=2)
    
    if not results:
        print("  ➜ Maaf, tidak ditemukan informasi yang relevan.")
    else:
        for i, res in enumerate(results, 1):
            source = res['metadata']['source']
            page = res['metadata']['page']
            snippet = res['text'][:300] + "..."
            print(f"  ➜ HASIL {i} (Sumber: {source}, Hal: {page})")
            print(f"     Teks: {snippet}\n")
    print("-" * 80)

HASIL PENCARIAN PRODUK MITSUBISHI (HYBRID SEARCH)

[QUERY 1]: Detail spesifikasi Mitsubishi Destinator
  ➜ HASIL 1 (Sumber: Mitsubishi-Xpander-2020-ID.pdf, Hal: 1)
     Teks: Semua fitur berteknologi canggih milik Mitsubishi Xpander menawarkan banyak kemudahan bagi Anda dan keluarga, yang akan memberikan pengalaman baru dalam berkendara.Advanced Technology & Safety Features RISE BODY (Reinforced Impact Safety Evolution) Teknologi rangka bodi yang kokoh dirancang khusus s...

  ➜ HASIL 2 (Sumber: Mitsubishi-L300-2019-ID.pdf, Hal: 2)
     Teks: . Mitsubishi L300 Pickup menyediakan beragam pilihan seperti kargo Standard dan FlatDeck yang dapat disesuaikan dengan kebutuhan usaha Anda...

--------------------------------------------------------------------------------

[QUERY 2]: Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity yang besar
  ➜ HASIL 1 (Sumber: Mitsubishi-Xpander-2024-ID.pdf, Hal: 7)
     Teks: Clean Air PM 2.5, pollen, etc Micron Air Filtration (PM 2.